In [87]:
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV

from xgboost import XGBRegressor

df = pd.read_parquet(
    "../data/processed/player_features.parquet"
)

matchup_features = [
    # Matchups
    'opp_points_allowed_avg_3',
    'opp_points_allowed_avg_5',
    'opp_points_allowed_trend'
]

In [88]:
# ----------- Receiving Feature List -----------

receiving_features = [
    # Receiving
    'targets_avg_3',
    'targets_avg_5',
    'rec_avg_3',
    'rec_avg_5',
    'rec_yards_avg_3',
    'rec_yards_avg_5',
    'target_share_avg_3',
    'target_share_avg_5',
    'rec_air_yards_avg_3',
    'rec_air_yards_avg_5',
    'air_yards_share_avg_3',
    'air_yards_share_avg_5',
    'rec_td_avg_3',
    'rec_td_avg_5',
    'rec_yards_after_catch_avg_3',
    'rec_yards_after_catch_avg_5',

    # Trends
    'targets_trend',
    'target_share_trend',
    'rec_yards_trend',
    'rec_air_yards_trend'
]


In [89]:
# ----------- WR Training Dataset -----------

wr_df = df[df['position'] == 'WR'].copy()

wr_features = receiving_features.copy()

X_wr = wr_df[wr_features]
y_wr = wr_df['fantasy_points_ppr']

valid_wr = X_wr.notna().all(axis=1)

X_wr = X_wr[valid_wr]
y_wr = y_wr[valid_wr]

In [90]:
# ----------- TE Training Dataset -----------

te_df = df[df['position'] == 'TE'].copy()

te_features = receiving_features.copy()

X_te = te_df[te_features]
y_te = te_df['fantasy_points_ppr']

valid_te = X_te.notna().all(axis=1)

X_te = X_te[valid_te]
y_te = y_te[valid_te]

In [91]:
# ----------- RB Training Dataset -----------

rb_df = df[df['position'] == 'RB'].copy()
rb_features = [
    # Rushing
    'carries_avg_3',
    'carries_avg_5',
    'rushing_yards_avg_3',
    'rushing_yards_avg_5',
    'rushing_tds_avg_3',
    'rushing_tds_avg_5',
    'opportunities_avg_3',
    'opportunities_avg_5',

    # Receiving
    'targets_avg_3',
    'targets_avg_5',
    'rec_avg_3',
    'rec_avg_5',
    'rec_yards_avg_3',
    'rec_yards_avg_5',
    'target_share_avg_3',
    'target_share_avg_5',
    'rec_td_avg_3',
    'rec_td_avg_5',

    # Trends
    'carries_trend',
    'rushing_yards_trend',
    'opportunities_trend',
    'targets_trend',
    'target_share_trend',
    'rec_yards_trend'
] + matchup_features

X_rb = rb_df[rb_features]
y_rb = rb_df['fantasy_points_ppr']

# Removing empty row that has no prior game history
valid_rb = X_rb.notna().all(axis=1)

X_rb = X_rb[valid_rb]
y_rb = y_rb[valid_rb]


In [92]:
# ----------- QB Training Dataset -----------

qb_df = df[df['position'] == 'QB'].copy()

qb_features = [
    # Passing
    'completions_avg_3',
    'completions_avg_5',
    'attempts_avg_3',
    'attempts_avg_5',
    'passing_yards_avg_3',
    'passing_yards_avg_5',
    'passing_tds_avg_3',
    'passing_tds_avg_5',
    'passing_int_avg_3',
    'passing_int_avg_5',
    'passing_air_yards_avg_3',
    'passing_air_yards_avg_5',
    'passing_first_downs_avg_3',
    'passing_first_downs_avg_5',

    # Rushing
    'carries_avg_3',
    'carries_avg_5',
    'rushing_yards_avg_3',
    'rushing_yards_avg_5',
    'rushing_tds_avg_3',
    'rushing_tds_avg_5',

    # Trends
    'attempts_trend',
    'passing_yards_trend',
    'passing_air_yards_trend',
    'carries_trend',
    'rushing_yards_trend'
]

X_qb = qb_df[qb_features]
y_qb = qb_df['fantasy_points_ppr']

valid_qb = X_qb.notna().all(axis=1)

X_qb = X_qb[valid_qb]
y_qb = y_qb[valid_qb]

In [93]:
# ----------- K Training Dataset -----------

k_df = df[df['position'] == 'K'].copy()

k_features = [
    'fg_att_avg_3',
    'fg_att_avg_5',
    'fg_made_avg_3',
    'fg_made_avg_5',
    'fg_long_avg_3',
    'fg_long_avg_5',
    'fg_made_50_59_avg_3',
    'fg_made_50_59_avg_5',
    'pat_att_avg_3',
    'pat_att_avg_5',
    'pat_made_avg_3',
    'pat_made_avg_5',
    'fg_att_trend',
    'fg_made_trend'
]

k_df['kicker_fantasy_points'] = (
    3 * k_df['fg_made'] +
    1 * k_df['pat_made']
)

X_k = k_df[k_features]
y_k = k_df['kicker_fantasy_points']

valid_k = X_k.notna().all(axis=1)

X_k = X_k[valid_k]
y_k = y_k[valid_k]


In [94]:
# ----------- WR Training, Validation, Test Split -----------

wr_seasons = wr_df.loc[X_wr.index, 'season']

train_mask_wr = wr_seasons <= 2023
val_mask_wr = wr_seasons == 2024
test_mask_wr = wr_seasons == 2025

X_train_wr = X_wr[train_mask_wr]
y_train_wr = y_wr[train_mask_wr]

X_val_wr = X_wr[val_mask_wr]
y_val_wr = y_wr[val_mask_wr]

X_test_wr = X_wr[test_mask_wr]
y_test_wr = y_wr[test_mask_wr]

print("WR train:", X_train_wr.shape)
print("WR validation:", X_val_wr.shape)
print("WR test:", X_test_wr.shape)

WR train: (10724, 23)
WR validation: (2205, 23)
WR test: (2270, 23)


In [95]:
# ----------- WR Linear Regression Model -----------

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

wr_linear_model = LinearRegression()

wr_linear_model.fit(
    X_train_wr,
    y_train_wr
)

y_val_pred_wr = wr_linear_model.predict(X_val_wr)

mae_wr = mean_absolute_error(
    y_val_wr,
    y_val_pred_wr
)

rmse_wr = root_mean_squared_error(
    y_val_wr,
    y_val_pred_wr
)

print("WR Linear Regression Validation MAE:", mae_wr)
print("WR Linear Regression Validation RMSE:", rmse_wr)

WR Linear Regression Validation MAE: 4.66211763317456
WR Linear Regression Validation RMSE: 6.420020243065681


In [96]:
# ----------- WR Random Forest Model -----------
from sklearn.ensemble import RandomForestRegressor

wr_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

wr_rf_model.fit(
    X_train_wr,
    y_train_wr
)

y_val_pred_rf_wr = wr_rf_model.predict(X_val_wr)

mae_rf_wr = mean_absolute_error(
    y_val_wr,
    y_val_pred_rf_wr
)

rmse_rf_wr = root_mean_squared_error(
    y_val_wr,
    y_val_pred_rf_wr
)

print("WR Random Forest Validation MAE:", mae_rf_wr)
print("WR Random Forest Validation RMSE:", rmse_rf_wr)

WR Random Forest Validation MAE: 4.829346930137134
WR Random Forest Validation RMSE: 6.536914919870213


In [97]:
# ----------- WR XGBoost Model -----------
from xgboost import XGBRegressor

wr_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

wr_xgb_model.fit(
    X_train_wr,
    y_train_wr
)

y_val_pred_xgb_wr = wr_xgb_model.predict(X_val_wr)

mae_xgb_wr = mean_absolute_error(
    y_val_wr,
    y_val_pred_xgb_wr
)

rmse_xgb_wr = root_mean_squared_error(
    y_val_wr,
    y_val_pred_xgb_wr
)

print("WR XGBoost Validation MAE:", mae_xgb_wr)
print("WR XGBoost Validation RMSE:", rmse_xgb_wr)

WR XGBoost Validation MAE: 5.042177961213181
WR XGBoost Validation RMSE: 6.898615759207582


In [98]:
# ----------- Tuning Models -----------

# XG Boost

wr_xgb_tuned_1 = XGBRegressor(
    n_estimators=300, # More trees
    learning_rate=0.05, # Smaller learning steps
    max_depth=3, # Shallower trees
    subsample=0.8, # Row sampling
    colsample_bytree=0.8, # feature sampling
    random_state=42
)

wr_xgb_tuned_1.fit(
    X_train_wr,
    y_train_wr
)

y_val_pred_xgb_tuned_wr = wr_xgb_tuned_1.predict(X_val_wr)

mae_xgb_tuned_wr = mean_absolute_error(
    y_val_wr,
    y_val_pred_xgb_tuned_wr
)

rmse_xgb_tuned_wr = root_mean_squared_error(
    y_val_wr,
    y_val_pred_xgb_tuned_wr
)

print("Tuned XGBoost MAE:", mae_xgb_tuned_wr)
print("Tuned XGBoost RMSE:", rmse_xgb_tuned_wr)

# Random Forest

wr_rf_tuned_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

wr_rf_tuned_1.fit(
    X_train_wr,
    y_train_wr
)

y_val_pred_rf_tuned_1 = wr_rf_tuned_1.predict(X_val_wr)

mae_rf_tuned_wr = mean_absolute_error(
    y_val_wr,
    y_val_pred_rf_tuned_1
)

rmse_rf_tuned_wr = root_mean_squared_error(
    y_val_wr,
    y_val_pred_rf_tuned_1
)

print("Tuned Random Forest MAE:", mae_rf_tuned_wr)
print("Tuned Random Forest RMSE:", rmse_rf_tuned_wr)

Tuned XGBoost MAE: 4.689625710082973
Tuned XGBoost RMSE: 6.436851682950997
Tuned Random Forest MAE: 4.700660364955156
Tuned Random Forest RMSE: 6.446492876962668


In [99]:
# ----------- TE Training, Validation, Test Split -----------

te_seasons = te_df.loc[X_te.index, 'season']

train_mask_te = te_seasons <= 2023
val_mask_te = te_seasons == 2024
test_mask_te = te_seasons == 2025

X_train_te = X_te[train_mask_te]
y_train_te = y_te[train_mask_te]

X_val_te = X_te[val_mask_te]
y_val_te = y_te[val_mask_te]

X_test_te = X_te[test_mask_te]
y_test_te = y_te[test_mask_te]

print("TE train:", X_train_te.shape)
print("TE validation:", X_val_te.shape)
print("TE test:", X_test_te.shape)

TE train: (5230, 23)
TE validation: (1096, 23)
TE test: (1150, 23)


In [100]:
# ----------- TE Linear Regression Model -----------

te_linear_model = LinearRegression()

te_linear_model.fit(
    X_train_te,
    y_train_te
)

y_val_pred_te = te_linear_model.predict(X_val_te)

mae_te = mean_absolute_error(
    y_val_te,
    y_val_pred_te
)

rmse_te = root_mean_squared_error(
    y_val_te,
    y_val_pred_te
)

print("TE Linear Regression Validation MAE:", mae_te)
print("TE Linear Regression Validation RMSE:", rmse_te)

TE Linear Regression Validation MAE: 3.746240932269298
TE Linear Regression Validation RMSE: 5.058150074505903


In [101]:
# ----------- TE Random Forest Model -----------

te_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

te_rf_model.fit(
    X_train_te,
    y_train_te
)

y_val_pred_rf_te = te_rf_model.predict(X_val_te)

mae_rf_te = mean_absolute_error(
    y_val_te,
    y_val_pred_rf_te
)

rmse_rf_te = root_mean_squared_error(
    y_val_te,
    y_val_pred_rf_te
)

print("TE Random Forest Validation MAE:", mae_rf_te)
print("TE Random Forest Validation RMSE:", rmse_rf_te)

TE Random Forest Validation MAE: 3.90450796836983
TE Random Forest Validation RMSE: 5.181108386027086


In [102]:
# ----------- TE XGBoost Model -----------

te_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

te_xgb_model.fit(
    X_train_te,
    y_train_te
)

y_val_pred_xgb_te = te_xgb_model.predict(X_val_te)

mae_xgb_te = mean_absolute_error(
    y_val_te,
    y_val_pred_xgb_te
)

rmse_xgb_te = root_mean_squared_error(
    y_val_te,
    y_val_pred_xgb_te
)

print("TE XGBoost Validation MAE:", mae_xgb_te)
print("TE XGBoost Validation RMSE:", rmse_xgb_te)

TE XGBoost Validation MAE: 3.9824659422768733
TE XGBoost Validation RMSE: 5.36673952006011


In [103]:
# ----------- RB Training, Validation, Test Split -----------

rb_seasons = rb_df.loc[X_rb.index, 'season']

train_mask_rb = rb_seasons <= 2023
val_mask_rb = rb_seasons == 2024
test_mask_rb = rb_seasons == 2025

X_train_rb = X_rb[train_mask_rb]
y_train_rb = y_rb[train_mask_rb]

X_val_rb = X_rb[val_mask_rb]
y_val_rb = y_rb[val_mask_rb]

X_test_rb = X_rb[test_mask_rb]
y_test_rb = y_rb[test_mask_rb]

print(X_train_rb.shape)
print(X_val_rb.shape)
print(X_test_rb.shape)

(6719, 27)
(1388, 27)
(1424, 27)


In [104]:
# ----------- RB Linear Regression Model -----------

rb_linear_model = LinearRegression()

rb_linear_model.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_rb = rb_linear_model.predict(X_val_rb)

mae_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_rb
)

rmse_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_rb
)

print("RB Linear Regression Validation MAE:", mae_rb)
print("RB Linear Regression Validation RMSE:", rmse_rb)

RB Linear Regression Validation MAE: 4.522561057140344
RB Linear Regression Validation RMSE: 6.1146600823076085


In [105]:
# ----------- RB Random Forest Model -----------

rb_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rb_rf_model.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_rf_rb = rb_rf_model.predict(X_val_rb)

mae_rf_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_rf_rb
)

rmse_rf_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_rf_rb
)

print("RB Random Forest Validation MAE:", mae_rf_rb)
print("RB Random Forest Validation RMSE:", rmse_rf_rb)

RB Random Forest Validation MAE: 4.736242002881844
RB Random Forest Validation RMSE: 6.307597257250862


In [106]:
# ----------- RB XGBoost Model -----------

rb_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

rb_xgb_model.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_xgb_rb = rb_xgb_model.predict(X_val_rb)

mae_xgb_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_xgb_rb
)

rmse_xgb_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_xgb_rb
)

print("RB XGBoost Validation MAE:", mae_xgb_rb)
print("RB XGBoost Validation RMSE:", rmse_xgb_rb)

RB XGBoost Validation MAE: 4.8546556000818395
RB XGBoost Validation RMSE: 6.600361159412439


In [107]:
# ----------- Tuning Models -----------

# Random Forest
rb_rf_tuned_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

rb_rf_tuned_1.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_rf_tuned_rb = rb_rf_tuned_1.predict(X_val_rb)

mae_rf_tuned_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_rf_tuned_rb
)

rmse_rf_tuned_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_rf_tuned_rb
)

print("Tuned RB Random Forest MAE:", mae_rf_tuned_rb)
print("Tuned RB Random Forest RMSE:", rmse_rf_tuned_rb)

# XGBoost
rb_xgb_tuned_1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

rb_xgb_tuned_1.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_xgb_tuned_rb = rb_xgb_tuned_1.predict(X_val_rb)

mae_xgb_tuned_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_xgb_tuned_rb
)

rmse_xgb_tuned_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_xgb_tuned_rb
)

print("tuned RB XGBoost MAE:", mae_xgb_tuned_rb)
print("Tuned RB XGBoost RMSE:", rmse_xgb_tuned_rb)

Tuned RB Random Forest MAE: 4.56880493770237
Tuned RB Random Forest RMSE: 6.158577223388805
tuned RB XGBoost MAE: 4.527890319393073
Tuned RB XGBoost RMSE: 6.130208789059104


In [108]:
# ----------- RB Chronological Ordering for Model Selection -----------

rb_train_info = rb_df.loc[
    X_train_rb.index,
    ['season', 'week']
].copy()

rb_chronological_order = rb_train_info.sort_values(
    ['season', 'week']
).index

X_train_rb_chrono = X_train_rb.loc[rb_chronological_order]
y_train_rb_chrono = y_train_rb.loc[rb_chronological_order]

In [109]:
# ----------- RB Expanding Season Validation Folds -----------

rb_season_series = rb_df.loc[
    X_train_rb_chrono.index,
    'season'
]

rb_cv_splits = []

for val_season in [2021, 2022, 2023]:
    train_indices = rb_season_series[rb_season_series < val_season].index
    val_indices = rb_season_series[rb_season_series == val_season].index

    train_positions = X_train_rb_chrono.index.get_indexer(train_indices)
    val_positions = X_train_rb_chrono.index.get_indexer(val_indices)

    rb_cv_splits.append((train_positions, val_positions))

for i, (train_idx, val_idx) in enumerate(rb_cv_splits, start=1):
    print(
        f"Fold {i}:",
        len(train_idx),
        "train rows,",
        len(val_idx),
        "validation rows"
    )

Fold 1: 2614 train rows, 1356 validation rows
Fold 2: 3970 train rows, 1423 validation rows
Fold 3: 5393 train rows, 1326 validation rows


In [110]:
# ----------- RB XGBoost Grid Search -----------

rb_param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.03, 0.05],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

rb_xgb_grid = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=rb_param_grid,
    scoring='neg_mean_absolute_error',
    cv=rb_cv_splits,
    n_jobs=-1
)

rb_xgb_grid.fit(
    X_train_rb_chrono,
    y_train_rb_chrono
)

print("Best RB parameters:", rb_xgb_grid.best_params_)
print("Best RB CV MAE:", -rb_xgb_grid.best_score_)

best_rb_xgb_model = rb_xgb_grid.best_estimator_

y_val_pred_best_rb_xgb = best_rb_xgb_model.predict(X_val_rb)

best_rb_xgb_val_mae = mean_absolute_error(
    y_val_rb,
    y_val_pred_best_rb_xgb
)

best_rb_xgb_val_rmse = root_mean_squared_error(
    y_val_rb,
    y_val_pred_best_rb_xgb
)

print("Best RB XGBoost 2024 Validation MAE:", best_rb_xgb_val_mae)
print("Best RB XGBoost 2024 Validation RMSE:", best_rb_xgb_val_rmse)

Best RB parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.8}
Best RB CV MAE: 4.8562097718016775
Best RB XGBoost 2024 Validation MAE: 4.538175625275466
Best RB XGBoost 2024 Validation RMSE: 6.142847995723407


In [111]:
# ----------- QB Training, Validation, Test Split -----------

qb_seasons = qb_df.loc[X_qb.index, 'season']

train_mask_qb = qb_seasons <= 2023
val_mask_qb = qb_seasons == 2024
test_mask_qb = qb_seasons == 2025

X_train_qb = X_qb[train_mask_qb]
y_train_qb = y_qb[train_mask_qb]

X_val_qb = X_qb[val_mask_qb]
y_val_qb = y_qb[val_mask_qb]

X_test_qb = X_qb[test_mask_qb]
y_test_qb = y_qb[test_mask_qb]

print(X_train_qb.shape)
print(X_val_qb.shape)
print(X_test_qb.shape)

(2764, 28)
(586, 28)
(583, 28)


In [112]:
# ----------- QB Linear Regression Model -----------

qb_linear_model = LinearRegression()

qb_linear_model.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_qb = qb_linear_model.predict(X_val_qb)

mae_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_qb
)

rmse_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_qb
)

print("QB Linear Regression Validation MAE:", mae_qb)
print("QB Linear Regression Validation RMSE:", rmse_qb)

QB Linear Regression Validation MAE: 6.256282005205791
QB Linear Regression Validation RMSE: 7.8977820053561585


In [113]:
# ----------- QB Random Forest Model -----------

qb_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

qb_rf_model.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_rf_qb = qb_rf_model.predict(X_val_qb)

mae_rf_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_rf_qb
)

rmse_rf_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_rf_qb
)

print("QB Random Forest Validation MAE:", mae_rf_qb)
print("QB Random Forest Validation RMSE:", rmse_rf_qb)

QB Random Forest Validation MAE: 6.295587030716724
QB Random Forest Validation RMSE: 7.982915151644903


In [114]:
# ----------- QB XGBoost Model -----------

qb_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

qb_xgb_model.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_xgb_qb = qb_xgb_model.predict(X_val_qb)

mae_xgb_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_xgb_qb
)

rmse_xgb_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_xgb_qb
)

print("QB XGBoost Validation MAE:", mae_xgb_qb)
print("QB XGBoost Validation RMSE:", rmse_xgb_qb)

QB XGBoost Validation MAE: 6.618043365555819
QB XGBoost Validation RMSE: 8.425786578354213


In [115]:
# ----------- QB Tuned Random Forest -----------

qb_rf_tuned_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

qb_rf_tuned_1.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_rf_tuned_qb = qb_rf_tuned_1.predict(X_val_qb)

mae_rf_tuned_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_rf_tuned_qb
)

rmse_rf_tuned_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_rf_tuned_qb
)

print("Tuned QB Random Forest MAE:", mae_rf_tuned_qb)
print("Tuned QB Random Forest RMSE:", rmse_rf_tuned_qb)

Tuned QB Random Forest MAE: 6.203933032024796
Tuned QB Random Forest RMSE: 7.891571694354197


In [116]:
# ----------- QB Tuned XGBoost -----------

qb_xgb_tuned_1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

qb_xgb_tuned_1.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_xgb_tuned_qb = qb_xgb_tuned_1.predict(X_val_qb)

mae_xgb_tuned_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_xgb_tuned_qb
)

rmse_xgb_tuned_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_xgb_tuned_qb
)

print("Tuned QB XGBoost MAE:", mae_xgb_tuned_qb)
print("Tuned QB XGBoost RMSE:", rmse_xgb_tuned_qb)

Tuned QB XGBoost MAE: 6.315460959091122
Tuned QB XGBoost RMSE: 8.000544106164316


In [117]:
# ----------- QB Chronological Ordering for Model Selection -----------

qb_train_info = qb_df.loc[
    X_train_qb.index,
    ['season', 'week']
].copy()

qb_chronological_order = qb_train_info.sort_values(
    ['season', 'week']
).index

X_train_qb_chrono = X_train_qb.loc[qb_chronological_order]
y_train_qb_chrono = y_train_qb.loc[qb_chronological_order]

In [118]:
# ----------- QB Expanding Season Validation Folds -----------

qb_season_series = qb_df.loc[
    X_train_qb_chrono.index,
    'season'
]

qb_cv_splits = []

for val_season in [2021, 2022, 2023]:
    train_indices = qb_season_series[qb_season_series < val_season].index
    val_indices = qb_season_series[qb_season_series == val_season].index

    train_positions = X_train_qb_chrono.index.get_indexer(train_indices)
    val_positions = X_train_qb_chrono.index.get_indexer(val_indices)

    qb_cv_splits.append((train_positions, val_positions))

for i, (train_idx, val_idx) in enumerate(qb_cv_splits, start=1):
    print(
        f"Fold {i}:",
        len(train_idx),
        "train rows,",
        len(val_idx),
        "validation rows"
    )

Fold 1: 1058 train rows, 574 validation rows
Fold 2: 1632 train rows, 550 validation rows
Fold 3: 2182 train rows, 582 validation rows


In [119]:
# ----------- QB Random Forest Grid Search -----------

qb_rf_param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [6, 10, 14],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 5],
    'max_features': [0.7, 1.0]
}

qb_rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=42
    ),
    param_grid=qb_rf_param_grid,
    scoring='neg_mean_absolute_error',
    cv=qb_cv_splits,
    n_jobs=-1
)

qb_rf_grid.fit(
    X_train_qb_chrono,
    y_train_qb_chrono
)

print("Best QB RF parameters:", qb_rf_grid.best_params_)
print("Best QB RF CV MAE:", -qb_rf_grid.best_score_)

best_qb_rf_model = qb_rf_grid.best_estimator_

y_val_pred_best_qb_rf = best_qb_rf_model.predict(X_val_qb)

best_qb_rf_val_mae = mean_absolute_error(
    y_val_qb,
    y_val_pred_best_qb_rf
)

best_qb_rf_val_rmse = root_mean_squared_error(
    y_val_qb,
    y_val_pred_best_qb_rf
)

print("Best QB Random Forest 2024 Validation MAE:", best_qb_rf_val_mae)
print("Best QB Random Forest 2024 Validation RMSE:", best_qb_rf_val_rmse)

Best QB RF parameters: {'max_depth': 6, 'max_features': 0.7, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best QB RF CV MAE: 6.070736287037978
Best QB Random Forest 2024 Validation MAE: 6.200709996891491
Best QB Random Forest 2024 Validation RMSE: 7.879520105293151


In [120]:
# ----------- K Training, Validation, Test Split -----------

k_seasons = k_df.loc[X_k.index, 'season']

train_mask_k = k_seasons <= 2023
val_mask_k = k_seasons == 2024
test_mask_k = k_seasons == 2025

X_train_k = X_k[train_mask_k]
y_train_k = y_k[train_mask_k]

X_val_k = X_k[val_mask_k]
y_val_k = y_k[val_mask_k]

X_test_k = X_k[test_mask_k]
y_test_k = y_k[test_mask_k]

print(X_train_k.shape)
print(X_val_k.shape)
print(X_test_k.shape)

(2370, 17)
(491, 17)
(496, 17)


In [121]:
# ----------- K Linear Regression Model -----------

k_linear_model = LinearRegression()

k_linear_model.fit(
    X_train_k,
    y_train_k
)

y_val_pred_k = k_linear_model.predict(X_val_k)

mae_k = mean_absolute_error(
    y_val_k,
    y_val_pred_k
)

rmse_k = root_mean_squared_error(
    y_val_k,
    y_val_pred_k
)

print("K Linear Regression Validation MAE:", mae_k)
print("K Linear Regression Validation RMSE:", rmse_k)

K Linear Regression Validation MAE: 3.0262149292623186
K Linear Regression Validation RMSE: 3.721555398892931


In [122]:
# ----------- K Random Forest Model -----------

k_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

k_rf_model.fit(
    X_train_k,
    y_train_k
)

y_val_pred_rf_k = k_rf_model.predict(X_val_k)

mae_rf_k = mean_absolute_error(
    y_val_k,
    y_val_pred_rf_k
)

rmse_rf_k = root_mean_squared_error(
    y_val_k,
    y_val_pred_rf_k
)

print("K Random Forest Validation MAE:", mae_rf_k)
print("K Random Forest Validation RMSE:", rmse_rf_k)

K Random Forest Validation MAE: 3.10224711473184
K Random Forest Validation RMSE: 3.8115686442999364


In [123]:
# ----------- K XGBoost Model -----------

k_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

k_xgb_model.fit(
    X_train_k,
    y_train_k
)

y_val_pred_xgb_k = k_xgb_model.predict(X_val_k)

mae_xgb_k = mean_absolute_error(
    y_val_k,
    y_val_pred_xgb_k
)

rmse_xgb_k = root_mean_squared_error(
    y_val_k,
    y_val_pred_xgb_k
)

print("K XGBoost Validation MAE:", mae_xgb_k)
print("K XGBoost Validation RMSE:", rmse_xgb_k)

K XGBoost Validation MAE: 3.363332509994507
K XGBoost Validation RMSE: 4.18718147277832
